# 01 - Business Entity Resolution: Exploratory Data Analysis (EDA)

This notebook performs a comprehensive, beginner-friendly exploratory analysis of the **Business Entity Resolution** dataset.
It explores data shapes, schema types, country distributions, ground-truth match cardinalities, text characteristics, and train vs. test distribution shifts without building ML models or normalizing text.

## Step 1: Imports

Importing essential modules required for data exploration.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import csv

# Set pandas display options for clean tabular visualization
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
print("Step 1 complete: Imports successful.")

Step 1 complete: Imports successful.


## Step 2: Load All Datasets

Locating and loading dataset paths for train and test TSV files using tab separation (`sep="\t"`).

In [2]:
# Flexible directory resolution for project root vs notebooks directory
DATASET_DIR = Path("dataset") if Path("dataset").exists() else Path("../dataset")
TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

file_paths = {
    "train_source1": TRAIN_DIR / "train_source1.tsv",
    "train_source2": TRAIN_DIR / "train_source2.tsv",
    "train_source3": TRAIN_DIR / "train_source3.tsv",
    "train_ground_truth": TRAIN_DIR / "train_ground_truth.tsv",
    "test_source1": TEST_DIR / "test_source1.tsv",
    "test_source2": TEST_DIR / "test_source2.tsv",
    "test_source3": TEST_DIR / "test_source3.tsv",
}

print(f"Dataset root resolved to: {DATASET_DIR.resolve()}")
for name, pth in file_paths.items():
    print(f" - {name}: {pth} (Exists: {pth.exists()})")

Dataset root resolved to: D:\Mydata\EpicGamez\business-entity-resolution\dataset
 - train_source1: dataset\train\train_source1.tsv (Exists: True)
 - train_source2: dataset\train\train_source2.tsv (Exists: True)
 - train_source3: dataset\train\train_source3.tsv (Exists: True)
 - train_ground_truth: dataset\train\train_ground_truth.tsv (Exists: True)
 - test_source1: dataset\test\test_source1.tsv (Exists: True)
 - test_source2: dataset\test\test_source2.tsv (Exists: True)
 - test_source3: dataset\test\test_source3.tsv (Exists: True)


## Step 3: Basic Inspection

Inspecting shape, column names, data types, missing-value counts, duplicate `entity_id` count, and unique entity IDs for every dataset.

In [3]:
dataset_summaries = []

for name, pth in file_paths.items():
    # Read header and first 5 rows for sample inspection
    df_head = pd.read_csv(pth, sep="\t", encoding="utf-8", dtype=str, nrows=5)
    
    # Read full file memory-efficiently with csv.reader for exact row counts and ID uniqueness
    total_rows = 0
    null_counts = {col: 0 for col in df_head.columns}
    id_col = df_head.columns[0]
    id_set = set()
    dup_ids = 0
    
    with open(pth, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader)
        for row in reader:
            total_rows += 1
            eid = row[0]
            if eid in id_set:
                dup_ids += 1
            else:
                id_set.add(eid)
                
            for idx, col_name in enumerate(header):
                val = row[idx] if idx < len(row) else ""
                if not val or val.lower() == "nan":
                    null_counts[col_name] += 1
                    
    dataset_summaries.append({
        "Dataset": name,
        "Rows": total_rows,
        "Columns": len(header),
        "Column Names": ", ".join(header),
        "Unique IDs": len(id_set),
        "Duplicate IDs": dup_ids,
        "Missing Values": null_counts
    })
    
    print(f"=== {name} ===")
    print(f"Shape: ({total_rows:,}, {len(header)})")
    print(f"Columns: {header}")
    print(f"Data Types: {{col: 'object/string' for col in header}}")
    print(f"Missing Values: {null_counts}")
    print(f"Duplicate {id_col} Count: {dup_ids}")
    print(f"Unique {id_col} Count: {len(id_set):,}")
    print("First 5 Rows:")
    display(df_head)
    print("-" * 60)

=== train_source1 ===
Shape: (2,206,821, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']
Data Types: {col: 'object/string' for col in header}
Missing Values: {'entity_id': 0, 'business_name': 0, 'business_address': 0, 'country': 0}
Duplicate entity_id Count: 0
Unique entity_id Count: 2,206,821
First 5 Rows:


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West Bengal",India


------------------------------------------------------------


=== train_source2 ===
Shape: (5,034,616, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']
Data Types: {col: 'object/string' for col in header}
Missing Values: {'entity_id': 0, 'business_name': 1, 'business_address': 168967, 'country': 0}
Duplicate entity_id Count: 0
Unique entity_id Count: 5,034,616
First 5 Rows:


,entity_id,business_name,business_address,country
0,S2-166376419,राम मार्केटिंग प्राइवेट लिमिटेड,"KH NO. -570/13, NEW DELHI, WEST DELHI, Delhi",India
1,S2-764573417,-- Holloway Peak Inc Seafood,"105 ELM ST, MORGANTON, NC",US
2,S2-639257739,आदित्य प्रॉपर्टीज एलएलपी,"G-3/571, GULMOHAR COLONY, BHOPAL, Madhya Pradesh",India
3,S2-163963287,Summit Inc,"GREENSBORO, NC, 19 1/2 STARDUST TRAIL",US
4,S2-49942811,Delta Tetlecommunication Inc,"914 PIERPONT AVE, CLEVELAND, OH",US


...

,entity_id,business_name,business_address,country
0,S2-192345572,Brahma Infosoft,"COIMATORE COLONY, HUNSUR TQMYSORE DIST., Karnataka",India
1,S2-566025912,Marina Ecole France Sarl,"63 R. DE DIEPPE, LILLE, Hauts-de-France",France
2,S2-158121477,SCI Ptit Àmicale,"18 RUE JEN ZAY, Dunkerque, Nord",France
3,S2-89663826,Apex Summit,"67 KENTUCKY ST, SALYERSVILLE, KY",US
4,S2-884102769,Fresh Truist,"8264 FILLY COURT, ROANOKE COUNTY, VA",US


------------------------------------------------------------


=== test_source3 ===
Shape: (5,082,316, 4)
Columns: ['entity_id', 'business_name', 'business_address', 'country']
Data Types: {col: 'object/string' for col in header}
Missing Values: {'entity_id': 0, 'business_name': 0, 'business_address': 136098, 'country': 0}
Duplicate entity_id Count: 0
Unique entity_id Count: 5,082,316
First 5 Rows:


,entity_id,business_name,business_address,country
0,S3-462677478,मॉडर्न फाइनेंस,"No 10 Enkay Square, 448A, Udyog Vihar Phase V, Gurugram, Gurgaon, HR",India
1,S3-374810425,Shri Sai Infratech Co,"3/115, East Delhi, DL",India
2,S3-198586129,Fractales Amis Groupe S.A.S,"23 Rue Icmre, La Teste-de-buch, Gironde",France
3,S3-10300249,Shri Supreme Consulting Private (Limited),"H.no 910 A 3503, Mumbai, महाराष्ट्र",India
4,S3-604980231,Prime Realty Ventures Public Limited,"G.t. Karnal Road, Industrial Area, New Delhi, null, A-68, दिल्ली",India


------------------------------------------------------------


## Step 4: Country Analysis

Displaying value counts of `country` across all sources. Country is treated as an open-set string.

In [4]:
country_summaries = {}

for name, pth in file_paths.items():
    df_head = pd.read_csv(pth, sep="\t", encoding="utf-8", dtype=str, nrows=1)
    if "country" in df_head.columns:
        counts = {}
        with open(pth, "r", encoding="utf-8") as f:
            reader = csv.reader(f, delimiter="\t")
            hdr = next(reader)
            c_idx = hdr.index("country")
            for r in reader:
                c_val = r[c_idx] if len(r) > c_idx else "MISSING"
                counts[c_val] = counts.get(c_val, 0) + 1
        country_summaries[name] = counts
        print(f"=== {name} Country Value Counts ===")
        display(pd.Series(counts, name="count"))
        print("-" * 50)

=== train_source1 Country Value Counts ===


US       1323633
India     883188
Name: count, dtype: int64

--------------------------------------------------


=== train_source2 Country Value Counts ===


India    2017799
US       3016817
Name: count, dtype: int64

...

India     2312565
France     703378
US        1871330
Name: count, dtype: int64

--------------------------------------------------


=== test_source3 Country Value Counts ===


India     2405000
France     731615
US        1945701
Name: count, dtype: int64

--------------------------------------------------


## Step 5: Ground-Truth Analysis

Analyzing `train_ground_truth.tsv` match cardinalities per Source 1 entity.

In [5]:
total_s1_gt = 0
num_0 = 0
num_1 = 0
num_2 = 0
num_3_plus = 0
max_matches = 0
total_positive_links = 0

gt_matches_map = {}

with open(file_paths["train_ground_truth"], "r", encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    header = next(reader)
    for row in reader:
        total_s1_gt += 1
        s1_id = row[0]
        raw_m = row[1] if len(row) > 1 else ""
        m_list = [m.strip() for m in raw_m.split(",") if m.strip()]
        gt_matches_map[s1_id] = m_list
        
        n_m = len(m_list)
        total_positive_links += n_m
        if n_m > max_matches:
            max_matches = n_m
            
        if n_m == 0:
            num_0 += 1
        elif n_m == 1:
            num_1 += 1
        elif n_m == 2:
            num_2 += 1
        else:
            num_3_plus += 1

avg_matches = total_positive_links / total_s1_gt

print(f"Total Source 1 Entities in Ground Truth: {total_s1_gt:,}")
print(f"Source 1 Entities with ZERO matches (Singletons): {num_0:,} ({num_0/total_s1_gt*100:.2f}%)")
print(f"Source 1 Entities with EXACTLY 1 match: {num_1:,} ({num_1/total_s1_gt*100:.2f}%)")
print(f"Source 1 Entities with EXACTLY 2 matches: {num_2:,} ({num_2/total_s1_gt*100:.2f}%)")
print(f"Source 1 Entities with 3+ matches: {num_3_plus:,} ({num_3_plus/total_s1_gt*100:.2f}%)")
print(f"Maximum number of matches for one S1 entity: {max_matches}")
print(f"Average number of matches per S1 entity: {avg_matches:.4f}")
print(f"Total positive S1-S2/S3 links: {total_positive_links:,}")

Total Source 1 Entities in Ground Truth: 2,206,821
Source 1 Entities with ZERO matches (Singletons): 123,247 (5.58%)
Source 1 Entities with EXACTLY 1 match: 119,157 (5.40%)
Source 1 Entities with EXACTLY 2 matches: 375,212 (17.00%)
Source 1 Entities with 3+ matches: 1,589,205 (72.01%)
Maximum number of matches for one S1 entity: 11
Average number of matches per S1 entity: 3.4613
Total positive S1-S2/S3 links: 7,638,365


## Step 6: Analyze Match Source

Categorizing positive ground-truth links by target source (Source 2 vs Source 3).

In [6]:
s2_matches_count = 0
s3_matches_count = 0
only_s2_count = 0
only_s3_count = 0
both_s2_s3_count = 0
singleton_count = 0

for s1_id, m_list in gt_matches_map.items():
    if not m_list:
        singleton_count += 1
        continue
    has_s2 = any(m.startswith("S2-") for m in m_list)
    has_s3 = any(m.startswith("S3-") for m in m_list)
    
    s2_matches_count += sum(1 for m in m_list if m.startswith("S2-"))
    s3_matches_count += sum(1 for m in m_list if m.startswith("S3-"))
    
    if has_s2 and not has_s3:
        only_s2_count += 1
    elif has_s3 and not has_s2:
        only_s3_count += 1
    elif has_s2 and has_s3:
        both_s2_s3_count += 1

print(f"Total Source 2 matches: {s2_matches_count:,}")
print(f"Total Source 3 matches: {s3_matches_count:,}")
print(f"Entities matching ONLY Source 2: {only_s2_count:,} ({only_s2_count/total_s1_gt*100:.2f}%)")
print(f"Entities matching ONLY Source 3: {only_s3_count:,} ({only_s3_count/total_s1_gt*100:.2f}%)")
print(f"Entities matching BOTH Source 2 and Source 3: {both_s2_s3_count:,} ({both_s2_s3_count/total_s1_gt*100:.2f}%)")
print(f"Singleton entities (0 matches): {singleton_count:,} ({singleton_count/total_s1_gt*100:.2f}%)")

Total Source 2 matches: 3,693,619
Total Source 3 matches: 3,944,746
Entities matching ONLY Source 2: 143,029 (6.48%)
Entities matching ONLY Source 3: 164,498 (7.45%)
Entities matching BOTH Source 2 and Source 3: 1,776,047 (80.48%)
Singleton entities (0 matches): 123,247 (5.58%)


## Step 7: Verify Integrity

Verifying ID existences across source tables, checking for unknown prefixes, and testing intra-list duplicate IDs.

In [7]:
# Extract entity ID sets for ground-truth validation
train_s1_ids = set()
with open(file_paths["train_source1"], "r", encoding="utf-8") as f:
    r = csv.reader(f, delimiter="\t")
    next(r)
    train_s1_ids = {row[0] for row in r}

train_s2_ids = set()
with open(file_paths["train_source2"], "r", encoding="utf-8") as f:
    r = csv.reader(f, delimiter="\t")
    next(r)
    train_s2_ids = {row[0] for row in r}

train_s3_ids = set()
with open(file_paths["train_source3"], "r", encoding="utf-8") as f:
    r = csv.reader(f, delimiter="\t")
    next(r)
    train_s3_ids = {row[0] for row in r}

gt_s1_missing = len(set(gt_matches_map.keys()) - train_s1_ids)
invalid_s2 = 0
invalid_s3 = 0
unknown_prefixes = 0
intra_list_dupes = 0

for s1_id, m_list in gt_matches_map.items():
    if len(m_list) != len(set(m_list)):
        intra_list_dupes += 1
    for m in m_list:
        if m.startswith("S2-"):
            if m not in train_s2_ids:
                invalid_s2 += 1
        elif m.startswith("S3-"):
            if m not in train_s3_ids:
                invalid_s3 += 1
        else:
            unknown_prefixes += 1

print(f"1. GT S1 IDs missing in train_source1: {gt_s1_missing}")
print(f"2. S2 matches missing in train_source2: {invalid_s2}")
print(f"3. S3 matches missing in train_source3: {invalid_s3}")
print(f"4. Unknown prefix IDs (not S2-/S3-): {unknown_prefixes}")
print(f"5. Rows with duplicate IDs inside matched_entity_ids list: {intra_list_dupes}")

1. GT S1 IDs missing in train_source1: 0
2. S2 matches missing in train_source2: 0
3. S3 matches missing in train_source3: 0
4. Unknown prefix IDs (not S2-/S3-): 0
5. Rows with duplicate IDs inside matched_entity_ids list: 0


## Step 8: Display Actual Examples

Displaying side-by-side examples of true matches, singleton entities, and multi-match S1 entities to observe string corruptions.

In [8]:
# Pick sample S1 IDs for each category
matched_s1_sample = [s1 for s1, m in gt_matches_map.items() if len(m) == 1][:10]
singleton_s1_sample = [s1 for s1, m in gt_matches_map.items() if len(m) == 0][:10]
multi_s1_sample = [s1 for s1, m in gt_matches_map.items() if len(m) >= 3][:3]

needed_ids = set(matched_s1_sample) | set(singleton_s1_sample) | set(multi_s1_sample)
for s1 in matched_s1_sample + multi_s1_sample:
    needed_ids.update(gt_matches_map[s1])

# Fetch entity details for needed IDs
entity_details = {}

for name in ["train_source1", "train_source2", "train_source3"]:
    with open(file_paths[name], "r", encoding="utf-8") as f:
        r = csv.reader(f, delimiter="\t")
        hdr = next(r)
        for row in r:
            eid = row[0]
            if eid in needed_ids:
                entity_details[eid] = {
                    "name": row[1] if len(row) > 1 else "",
                    "address": row[2] if len(row) > 2 else "",
                    "country": row[3] if len(row) > 3 else ""
                }

# 1. True Match Examples Side-by-Side
side_by_side = []
for s1 in matched_s1_sample:
    m = gt_matches_map[s1][0]
    s1_info = entity_details.get(s1, {})
    m_info = entity_details.get(m, {})
    side_by_side.append({
        "S1 entity_id": s1,
        "S1 business_name": s1_info.get("name"),
        "S1 business_address": s1_info.get("address"),
        "S1 country": s1_info.get("country"),
        "matched entity_id": m,
        "matched business_name": m_info.get("name"),
        "matched business_address": m_info.get("address"),
        "matched country": m_info.get("country"),
    })

print("=== 10 True Matching Examples (Side-by-Side) ===")
display(pd.DataFrame(side_by_side))

# 2. Singleton Examples
singleton_rows = []
for s1 in singleton_s1_sample:
    s1_info = entity_details.get(s1, {})
    singleton_rows.append({
        "S1 entity_id": s1,
        "S1 business_name": s1_info.get("name"),
        "S1 business_address": s1_info.get("address"),
        "S1 country": s1_info.get("country"),
    })

print("\n=== 10 Singleton (Unmatched S1) Examples ===")
display(pd.DataFrame(singleton_rows))

# 3. Multi-match Examples
multi_rows = []
for s1 in multi_s1_sample:
    s1_info = entity_details.get(s1, {})
    for m in gt_matches_map[s1]:
        m_info = entity_details.get(m, {})
        multi_rows.append({
            "S1 entity_id": s1,
            "S1 business_name": s1_info.get("name"),
            "S1 business_address": s1_info.get("address"),
            "matched entity_id": m,
            "matched business_name": m_info.get("name"),
            "matched business_address": m_info.get("address"),
        })

print("\n=== Multi-Match S1 Examples (3+ matches) ===")
display(pd.DataFrame(multi_rows))

=== 10 True Matching Examples (Side-by-Side) ===


,S1 entity_id,S1 business_name,S1 business_address,S1 country,matched entity_id,matched business_name,matched business_address,matched country
0,S1-116043204,Red Consultants Pvt. Ltd.,"C/O Tapas Kumar Betal, Bhogpur, Purba Medinipur, Panskura, East Midnapore, West Bengal",India,S3-85523430,Red Pvt. Ltd. Center,,India
1,S1-473377609,Wave & Brothers Ltd,"Plot No. 65, Mahada Colony, Sai Nagar, Nagpur, Maharashtra",India,S3-433876173,Belobrixlum,"Door No 65, Mahada Colony, Sai Nagar, Nagpur, MH",India
2,S1-439203009,Ace Producer Private Limited,"H No 847 S No 113/12 Opp Patil Enclave, Pune, Maharashtra",India,S3-967165288,Ace Producer,"847 S No 113/12 Opp Patil Enclave, Pune, MH",India
3,S1-973290215,Gulf Ministries V Associates,"WI, Village Of Weston, 5504 Jamar Street",US,S3-925530888,Gulf V Ministries Associates,"5504. Jamar St, Schoield CITY, Wisconsin",US
4,S1-649259801,Guru Solutions Pvt Ltd,"443 Kantharaju Urs Road T K Layout, Mysore, Karnataka",India,S2-654066445,ಗುರು ಸೊಲ್ಯೂಷನ್ಸ್ ಪ್ರೈವೇಟ್ ಲಿಮಿಟೆಡ್,"C-443 KANTHARAJU URS ROAD T K LAYOUT, MYSORE, Karnataka",India
5,S1-862038247,Waters & Millar Aimei P.C.,"6267 Passing Sky Drive, Security-widefield, CO",US,S2-451274541,Aimei Waters & Millar P.C.,"6267B PASSING SKY DR, SECURITY-WIDEFIELD, CO",US
6,S1-89774084,Ap Infocom Private Limited,"14/307, Third Floor, Regency Appartment Johri Farm, Noor Nagar Extn., Delhi",India,S2-198723291,Ap Infocom Private [Ltd],"##14/307, THIRD FLOOR, REGENCY APPARTMENT JOHRI FARM, NOOR NAGAR EXTN., Delhi",India
7,S1-424050882,Ram Foods LLP,"Haryana, Cabin No.1, 2Nd Floor, Sector 16, Panchkula, Sco No. 18, Panchkula",India,S3-37198119,Ram LLP Service Service,"Sco No. 18, Panchkula, HR",India
8,S1-971813156,Education Center,"3025 Sanford Road, Atwater, OH",US,S3-394463058,Education Center,"3025 Sanford Rd, Rootstown Twp, Ohio",US
9,S1-106072003,Anchor Vista Inc,"Burnsville, MN, 2601 Valley View Drive",US,S2-699555547,Anchor Vista Incorporated,"2601 VALLEY VIEW DRIVE, BURNSVILLE, MN",US



=== 10 Singleton (Unmatched S1) Examples ===


,S1 entity_id,S1 business_name,S1 business_address,S1 country
0,S1-302869473,International Automation Consultants Inc,"329 Rev Walton Drive, Lockport, IL",US
1,S1-262997549,Gabriella's Preferred Security,"1013 Girard Avenue, Indianola, IA",US
2,S1-508022910,Twyla's Liquor Corp,"320 Flannery Lane, Silver Spring, MD",US
3,S1-666499407,Vadapalani Projects Pvt. Ltd.,"No.75, Natarajan Street, Dhanalakshmi Colony, Vadapalani, Chennai, Tamil Nadu",India
4,S1-965524997,Campbell Property Solutions,"25 Crimson View Drive, Sedona, AZ",US
5,S1-830177070,Namma Brand Pvt Ltd,"Aarjees Buildingssasthancoil Road Thycaud, Trivandrum, Palakkad, Kerala",India
6,S1-202262115,Mercado Guardian,"VA, Portsmouth City, 401 Stratford Street",US
7,S1-625699510,Engracia Fuhrmann Wells LLC,"3452 Brinkley Road, Unit 201, Temple Hills, MD",US
8,S1-604865181,Gold It Private Limited,"123, First Floor, Sahjanand Park, Nr. Swaminarayan Temple, Shahibaug, Ahmedabad, Gujarat",India
9,S1-88224713,Seaways & Associates,"Plot No : 318 Village Baska, Taluka - Halol, Vadodara, Panch Mahals, Gujarat",India



=== Multi-Match S1 Examples (3+ matches) ===


,S1 entity_id,S1 business_name,S1 business_address,matched entity_id,matched business_name,matched business_address
0,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S2-681193310,Maure Wilblims Colombier Inc,
1,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S2-743505751,Maure Williams Colombier,
2,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S3-775321672,Dréxkor,"85 Wanye Avenue, Ticonderoga Townshiip, New York"
3,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S3-11291185,maurewilliamscolombier.com,"Wayne Ave, Ticonderoga Townshiip, New York"
4,S1-965667,Maure Williams Colombier Inc,"85 Wayne Avenue, Ticonderoga, NY",S3-860443364,Maure Williams Inc Center,
5,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu",S2-249013014,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu"
6,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu",S2-197070651,Raj Investments LLP,"6(29), C.I.T. COLONY, 2ND MAIN ROAD MYLAPORE, CHENNAI, Tamil Nadu"
7,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu",S3-478195123,Raj Investments எல்எல்பி,"6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, TN"
8,S1-55344266,Raj Investments LLP,"6(29), C.I.T. Colony, 2Nd Main Road Mylapore, Chennai, Tamil Nadu",S3-384364074,ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி,"6(29), C.i.t. Colony, 2Nd Main Road Mylapore, Chennai, தமிழ்நாடு"
9,S1-343815751,Dahlia Power Reliable Scientific LLC,"630 45th Terrace, Kansas City, MO",S2-790675320,Dahlia Power Reliable,"KANSAS CITY, MO, 630 45ND TERRACE, null"


## Step 9: Basic Text Statistics

Calculating missing percentage, average string length, median string length, exact duplicates, and unique values for name and address fields across all datasets.

In [9]:
text_stats_table = []

for name in ["train_source1", "train_source2", "train_source3", "test_source1", "test_source2", "test_source3"]:
    pth = file_paths[name]
    n_lens, a_lens = [], []
    n_null, a_null = 0, 0
    n_unique, a_unique = set(), set()
    total_r = 0
    
    with open(pth, "r", encoding="utf-8") as f:
        reader = csv.reader(f, delimiter="\t")
        hdr = next(reader)
        n_idx = hdr.index("business_name")
        a_idx = hdr.index("business_address")
        for r in reader:
            total_r += 1
            n_val = r[n_idx].strip() if len(r) > n_idx else ""
            a_val = r[a_idx].strip() if len(r) > a_idx else ""
            
            if not n_val:
                n_null += 1
            else:
                n_lens.append(len(n_val))
                n_unique.add(n_val)
                
            if not a_val:
                a_null += 1
            else:
                a_lens.append(len(a_val))
                a_unique.add(a_val)
                
    text_stats_table.append({
        "Dataset": name,
        "Name Missing %": f"{n_null / total_r * 100:.2f}%",
        "Name Avg Len": f"{np.mean(n_lens):.2f}" if n_lens else "0.00",
        "Name Med Len": float(np.median(n_lens)) if n_lens else 0,
        "Name Unique": len(n_unique),
        "Name Exact Dups": (total_r - n_null) - len(n_unique),
        "Addr Missing %": f"{a_null / total_r * 100:.2f}%",
        "Addr Avg Len": f"{np.mean(a_lens):.2f}" if a_lens else "0.00",
        "Addr Med Len": float(np.median(a_lens)) if a_lens else 0,
        "Addr Unique": len(a_unique),
        "Addr Exact Dups": (total_r - a_null) - len(a_unique),
    })

display(pd.DataFrame(text_stats_table))

,Dataset,Name Missing %,Name Avg Len,Name Med Len,Name Unique,Name Exact Dups,Addr Missing %,Addr Avg Len,Addr Med Len,Addr Unique,Addr Exact Dups
0,train_source1,0.00%,24.03,24.0,1539229,667592,0.00%,52.07,41.0,2130606,76215
1,train_source2,0.00%,25.10,25.0,4402009,632607,3.36%,47.83,37.0,4337261,528388
2,train_source3,0.00%,25.20,25.0,4651609,633994,3.33%,48.32,42.0,4632764,476923
3,test_source1,0.00%,23.84,24.0,1238867,493677,0.00%,57.21,50.0,1677483,55061
4,test_source2,0.00%,25.70,25.0,4311041,576232,2.65%,51.78,43.0,4224783,533082
5,test_source3,0.00%,25.66,25.0,4521929,560387,2.68%,50.08,44.0,4456435,489783


## Step 10: Train vs Test Comparison

Comparing record counts, countries, missing-value rates, and average string lengths between training and test sources.
Highlighting countries appearing in test that do not appear in training.

In [10]:
comp_table = []

for src in ["1", "2", "3"]:
    tr_name = f"train_source{src}"
    te_name = f"test_source{src}"
    
    tr_info = next(item for item in dataset_summaries if item["Dataset"] == tr_name)
    te_info = next(item for item in dataset_summaries if item["Dataset"] == te_name)
    
    tr_countries = set(country_summaries.get(tr_name, {}).keys())
    te_countries = set(country_summaries.get(te_name, {}).keys())
    unseen_c = te_countries - tr_countries
    
    tr_ts = next(item for item in text_stats_table if item["Dataset"] == tr_name)
    te_ts = next(item for item in text_stats_table if item["Dataset"] == te_name)
    
    comp_table.append({
        "Source": f"Source {src}",
        "Train Records": f"{tr_info['Rows']:,}",
        "Test Records": f"{te_info['Rows']:,}",
        "Train Countries": ", ".join(sorted(tr_countries)),
        "Test Countries": ", ".join(sorted(te_countries)),
        "Unseen Test Countries": ", ".join(sorted(unseen_c)) if unseen_c else "None",
        "Train Name Miss %": tr_ts["Name Missing %"],
        "Test Name Miss %": te_ts["Name Missing %"],
        "Train Addr Miss %": tr_ts["Addr Missing %"],
        "Test Addr Miss %": te_ts["Addr Missing %"],
        "Train Avg Name Len": tr_ts["Name Avg Len"],
        "Test Avg Name Len": te_ts["Name Avg Len"],
        "Train Avg Addr Len": tr_ts["Addr Avg Len"],
        "Test Avg Addr Len": te_ts["Addr Avg Len"],
    })

display(pd.DataFrame(comp_table))

,Source,Train Records,Test Records,Train Countries,Test Countries,Unseen Test Countries,Train Name Miss %,Test Name Miss %,Train Addr Miss %,Test Addr Miss %,Train Avg Name Len,Test Avg Name Len,Train Avg Addr Len,Test Avg Addr Len
0,Source 1,"2,206,821","1,732,544","India, US","France, India, US",France,0.00%,0.00%,0.00%,0.00%,24.03,23.84,52.07,57.21
1,Source 2,"5,034,616","4,887,273","India, US","France, India, US",France,0.00%,0.00%,3.36%,2.65%,25.10,25.70,47.83,51.78
2,Source 3,"5,285,603","5,082,316","India, US","France, India, US",France,0.00%,0.00%,3.33%,2.68%,25.20,25.66,48.32,50.08


## DATASET SUMMARY

Key empirical numerical findings across all 7 dataset files:

1. **Dataset Record Counts**:
   - **Train Source 1**: 2,206,821 rows (2,206,821 unique IDs)
   - **Train Source 2**: 5,034,616 rows (5,034,616 unique IDs)
   - **Train Source 3**: 5,285,603 rows (5,285,603 unique IDs)
   - **Train Ground Truth**: 2,206,821 S1 entities
   - **Test Source 1**: 1,732,544 rows (1,732,544 unique IDs)
   - **Test Source 2**: 4,887,273 rows (4,887,273 unique IDs)
   - **Test Source 3**: 5,082,316 rows (5,082,316 unique IDs)

2. **Ground-Truth Cardinality & Linkages**:
   - **Singletons (0 matches)**: 123,247 S1 entities (5.58%)
   - **Exactly 1 match**: 119,157 S1 entities (5.40%)
   - **Exactly 2 matches**: 375,212 S1 entities (17.00%)
   - **3+ matches**: 1,589,205 S1 entities (72.01%)
   - **Maximum matches for one S1 entity**: 11
   - **Average matches per S1 entity**: 3.4613
   - **Total Positive S1-S2/S3 Links**: 7,638,365 (3,693,619 S2 links + 3,944,746 S3 links)
   - **Entities matching BOTH S2 and S3**: 1,776,047 S1 entities (80.48%)

3. **Missing Value Rates**:
   - `business_name`: 0.00% missing across all 6 sources (except 2 in train S2, 13 in train S3, 46 in test S2, 59 in test S3).
   - `business_address`: 0.00% missing in Source 1; ~3.34% missing in Train S2/S3; ~2.66% missing in Test S2/S3.

4. **Country & Open-Set Strings**:
   - Training sources contain only **US** and **India**.
   - Test sources contain **US**, **India**, and **France** (259,452 in Test S1; 703,378 in Test S2; 731,615 in Test S3).

5. **Data Integrity**:
   - 100% of ground-truth S1 IDs, S2 matched IDs, and S3 matched IDs exist in their corresponding source tables.
   - Zero malformed prefixes or intra-list duplicate IDs.

## OBSERVATIONS

Direct empirical data observations:

1. **Multi-Match Cardinality (1-to-Many)**: Over 72% of Source 1 entities match 3 or more entities across Source 2 and Source 3, with up to 11 matches per S1 entity.
2. **High Source Coverage**: 80.48% of S1 entities match records in BOTH Source 2 and Source 3 simultaneously.
3. **Low Singleton Rate in Train**: Only 5.58% of S1 entities in training have zero matches.
4. **Open-Set Geographic Drift**: The test dataset introduces a new country code (`France`) that never appears anywhere in the training data, emphasizing that country-level filtering must treat country as an open-set string.
5. **Exact Match Incompleteness**: High numbers of exact string duplicates exist for names and addresses, but variations in formatting, abbreviations, and street names require fuzzy string matching.